# 05 · Coastal strike probability + report figures
Monte Carlo: the track forecast + 1,000 error sequences sampled from real validation errors → share of
simulated tracks passing **within 100 km of each district's coastline**, cumulative windows 0–6/12/18/24 h.

Validation is done on the calibration years (`C.CAL`, 2019–2020, incl. Amphan), which the error bank (from the validation
years `C.VAL`, 2017–2018) has not seen.
Put a coastal district boundary file at `C.DISTRICTS_FILE` for accurate results; otherwise built-in coastal
reference points are used.

**Runtime:** CPU. **Takes:** ~10–30 min.

In [ ]:
# ▶ Setup: connect Google Drive, install missing packages, load the project code
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "lightgbm", "xgboost"], check=False)

PROJECT_DIR = "/content/drive/MyDrive/CycloneProject"      # <- change only if you put the project elsewhere
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import numpy as np, pandas as pd
from src import config as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("project:", C.PROJECT, "| ERA5:", C.ERA5_ROOT)

In [ ]:
import json, joblib
from src import strike as S, metrics as M, plots as P, tracks
from src.common import load_table
from src.labels import TARGET_TRACK

df = load_table(C.out("tables") / "storm_fixes")
FEATS = json.load(open(C.out("models") / "storm_features.json"))
TM = {t: joblib.load(C.out("models") / f"{t}.joblib") for t in TARGET_TRACK}
ERR = np.load(C.out("models") / "track_error_bank.npy")
DIST, src_kind = S.load_districts()
fixes = tracks.load_ibtracs(); TRK, INFO = tracks.build_tracks(fixes)
rng = np.random.default_rng(C.RANDOM_SEED)
print(f"{len(DIST)} districts from {src_kind} | error bank {len(ERR)} sequences")

def forecast(rows):
    return {t: TM[t].predict(rows[FEATS]) for t in TARGET_TRACK}

## Validation of strike probabilities (calibration years 2019–2020)

In [ ]:
ev = df[(df.split == "cal")].copy()
ev = ev[ev["c_dist_land_km"] < 900] if "c_dist_land_km" in ev else ev
ev = ev.iloc[::2]                                         # every 6 h is enough
fc = forecast(ev)
out = []
for i, (idx, r) in enumerate(ev.iterrows()):
    pd_lat = [fc[f"y_dlat_{L}"][i] for L in C.LEADS_H]; pd_lon = [fc[f"y_dlon_{L}"][i] for L in C.LEADS_H]
    hrs, HL, HN = S.simulate(r.lat, r.lon, pd_lat, pd_lon, ERR, C.N_MC_TRACKS_VALIDATION, rng)
    p = S.strike_probabilities(hrs, HL, HN, DIST)
    o = S.observed_hits(TRK[r.SID], r.time, DIST)
    out.append(p.merge(o, on="district").assign(SID=r.SID, time=r.time))
sv = pd.concat(out, ignore_index=True)
sv.to_csv(C.out("results") / "strike_validation_raw.csv", index=False)
rows, rel = [], {}
for L in C.LEADS_H:
    s = sv.dropna(subset=[f"o_{L}h"])
    rows.append(dict(window=f"0-{L} h", pairs=len(s), hits=int(s[f"o_{L}h"].sum()),
                     brier=M.brier_binary(s[f"o_{L}h"], s[f"p_{L}h"]),
                     brier_climatology=M.brier_binary(s[f"o_{L}h"], np.full(len(s), s[f"o_{L}h"].mean()))))
    rel[f"0-{L} h"] = M.reliability(s[f"o_{L}h"], s[f"p_{L}h"], bins=5)
sres = pd.DataFrame(rows); sres.to_csv(C.out("results") / "strike_validation.csv", index=False); display(sres.round(4))
P.reliability_plot(rel, f"Coastal strike probability - reliability ({C.span(C.CAL)})", C.out("figures") / "strike_reliability.png")

## Case study: a named storm
Change `C.CASE_STORM` to any storm name (e.g. `MOCHA` 2023, `BIPARJOY` 2023, `YAAS` 2021 - test years, unseen in training;
or `MONTHA` / `DITWAH` 2025 - the recent test storms).

In [ ]:
from src import io_era5
from src.common import buffer_grid
sid = INFO[INFO.NAME.str.upper() == C.CASE_STORM.upper()].SID
assert len(sid), f"{C.CASE_STORM} not found"
sid = sid.iloc[0]
rows = df[df.SID == sid].sort_values("time")
fc = forecast(rows)
forecasts = []
for i, (_, r) in enumerate(rows.iterrows()):
    if i % 4:
        continue                                         # a forecast every 12 h
    forecasts.append((r.time, r.lat, r.lon, [r.lat + fc[f"y_dlat_{L}"][i] for L in C.LEADS_H],
                      [r.lon + fc[f"y_dlon_{L}"][i] for L in C.LEADS_H]))
# strike map from the forecast issued ~24 h before the closest approach to land
k = max(0, int(np.argmin(rows["c_dist_land_km"].values)) - 8) if "c_dist_land_km" in rows else 0
r = rows.iloc[k]
hrs, HL, HN = S.simulate(r.lat, r.lon, [fc[f"y_dlat_{L}"][k] for L in C.LEADS_H],
                         [fc[f"y_dlon_{L}"][k] for L in C.LEADS_H], ERR, C.N_MC_TRACKS, rng)
sp = S.strike_probabilities(hrs, HL, HN, DIST)
sp["p"] = sp["p_24h"]
pts = {d["name"]: d["pts"].mean(axis=0) for d in DIST}
sp["lat"] = [pts[n][0] for n in sp.district]; sp["lon"] = [pts[n][1] for n in sp.district]
shown = sp[sp.p_24h > 0]
lat, lon = buffer_grid()
P.case_study_map(io_era5.load_lsm(), lat, lon, TRK[sid], forecasts,
                 f"{C.CASE_STORM}: observed track, 24 h forecasts and 0-24 h strike % (issued {r.time:%d %b %H} UTC)",
                 C.out("figures") / f"case_{C.CASE_STORM}.png", strike=shown)
table = sp[sp.p_24h > 0].sort_values("p_24h", ascending=False).copy()
for L in C.LEADS_H:
    table[f"{L} h"] = table[f"p_{L}h"].map(S.rounded_label)
table["colour (24 h)"] = (100 * table.p_24h).map(S.band)
display(table[["district", "state"] + [f"{L} h" for L in C.LEADS_H] + ["colour (24 h)"]])
table.to_csv(C.out("results") / f"case_{C.CASE_STORM}_strike_table.csv", index=False)

## Everything for the PPT

In [ ]:
import glob
print("Result tables:"); [print("  ", p) for p in sorted(glob.glob(str(C.out("results") / "*.csv")))]
print("Figures:"); [print("  ", p) for p in sorted(glob.glob(str(C.out("figures") / "*.png")))]